In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRanker

In [7]:
# ---------------------
# ① データ読み込み
# ---------------------
transactions = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv")
customers = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/customers.csv")
articles = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/articles.csv")
sample_submission = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv")

In [11]:
# transactions から特徴量生成の一例
features = transactions.copy()
features['t_dat'] = pd.to_datetime(features['t_dat'])

# 必要に応じて articles/customers を結合
data = features.merge(articles[['article_id', 'product_type_no']], on='article_id', how='left')
data = data.merge(customers[['customer_id', 'age']], on='customer_id', how='left')


In [12]:
features.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [13]:
data.head()

,t_dat,customer_id,article_id,price,sales_channel_id,product_type_no,age
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,283,24.0
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,306,24.0
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,252,32.0
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,252,32.0
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,252,32.0


In [14]:
# ---------------------
# ① データ読み込みと前処理
# ---------------------
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

# ② 正例データ（直近30日）
positive = transactions[transactions['t_dat'] >= '2020-09-15'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

# ③ 負例データ（正例の5倍）
np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*5),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*5),
})
neg['target'] = 0

# ④ 正例と重複しない負例のみ残す
neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index
)]

# ⑤ 正負例の結合
data = pd.concat([positive, neg], ignore_index=True)

# ⑥ transactionsから特徴量（例：days_ago）
trans_copy = transactions.copy()
trans_copy['t_dat'] = pd.to_datetime(trans_copy['t_dat'])
trans_copy['days_ago'] = (trans_copy['t_dat'].max() - trans_copy['t_dat']).dt.days

# 顧客×商品の最新購入日、購入頻度、最短days_ago
features = (
    trans_copy
    .groupby(['customer_id', 'article_id'])
    .agg(
        purchase_count=('t_dat', 'count'),
        last_purchase=('t_dat', 'max'),
        recency=('days_ago', 'min')
    )
    .reset_index()
)

# recency_weight を追加（時間減衰的な特徴）
features['recency_weight'] = 1 / np.log1p(features['recency'])

# ⑦ 特徴量と正負例を結合
data = data.merge(features, on=['customer_id', 'article_id'], how='left')

# 必要な属性をarticles/customersから追加
data = data.merge(articles[['article_id', 'product_type_no']], on='article_id', how='left')
data = data.merge(customers[['customer_id', 'age']], on='customer_id', how='left')

# ⑧ 特徴量整形
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].astype('category')

# ---------------------
# ⑨ 学習用データの整備
# ---------------------
# 顧客単位にソート（group対応）
data_sorted = data.sort_values('customer_id').reset_index(drop=True)

# 特徴量と目的変数
X = data_sorted.drop(columns=['customer_id', 'article_id', 'target'])
y = data_sorted['target']
groups = data_sorted.groupby('customer_id', observed=True).size().tolist()

# ---------------------
# ⑩ モデル学習（LGBMRanker）
# ---------------------
ranker = LGBMRanker(
    objective='lambdarank',
    metric='map',
    boosting_type='gbdt',
    random_state=42,
    n_estimators=100
)

ranker.fit(
    X, y,
    group=groups
)

# ---------------------
# ⑪ 予測と12件推薦
# ---------------------
data_sorted['pred_score'] = ranker.predict(X)

# 各顧客ごとにスコア上位12件を取得
top12 = (
    data_sorted
    .groupby('customer_id')
    .apply(lambda x: x.sort_values('pred_score', ascending=False).head(12))
    .reset_index(drop=True)
)

# ゼロ埋めと形式変換
top12['article_id'] = top12['article_id'].astype(str).str.zfill(10)

# 提出形式に整形
submission_df = (
    top12.groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x))
    .reset_index()
)


DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float32DType'>)

In [ ]:
# ---------------------
# ⑤ 年齢層別の人気商品を作成
# ---------------------
customers = customers[~customers['age'].isna()].copy()
customers['age_group3'] = pd.cut(customers['age'], bins=[0, 30, 60, 200], labels=['~30', '31-60', '60~'])

transactions = transactions.merge(customers[['customer_id', 'age_group3']], on='customer_id', how='left')
recent = transactions[transactions['t_dat'] >= '2020-09-14']

top_items_by_age = (
    recent.groupby('age_group3')['article_id']
    .value_counts()
    .groupby(level=0)
    .head(50)
    .reset_index()
    .rename(columns={'article_id': 'article_id_raw'})
)
top_items_by_age['article_id'] = top_items_by_age['article_id_raw'].astype(str).str.zfill(10)
age_group_popular_dict = top_items_by_age.groupby('age_group3')['article_id'].apply(list).to_dict()

# fallback（全体で人気な商品リスト）
popular_articles_100 = transactions['article_id'].value_counts().head(100).index.astype(str).str.zfill(10).tolist()

# ---------------------
# ⑥ 補完処理の適用
# ---------------------
top12['article_id'] = top12['article_id'].astype(str).str.zfill(10)
recommendations_df = (
    top12.groupby('customer_id')['article_id']
    .apply(list)
    .reset_index()
)

# 年齢層をマージ
age_map = customers[['customer_id', 'age_group3']]
recommendations_df = recommendations_df.merge(age_map, on='customer_id', how='left')

# 補完関数
def complete_with_agegroup(row):
    article_list = row['article_id']
    age_group = row['age_group3']
    if len(article_list) >= 12:
        return article_list[:12]
    fallback = age_group_popular_dict.get(age_group, popular_articles_100)
    fill_items = [a for a in fallback if a not in article_list]
    return article_list + fill_items[:12 - len(article_list)]

# 補完の実行
recommendations_df['prediction'] = recommendations_df.apply(complete_with_agegroup, axis=1)
recommendations_df['prediction'] = recommendations_df['prediction'].apply(lambda x: ' '.join(x))

print("✅ 年齢層ごとの補完を反映済みの推薦リストが完成しました")
recommendations_df.head()

In [ ]:
# 提出ファイル作成（欠損には人気記事を補完）
sample_submission = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv")

submission = sample_submission[['customer_id']].merge(submission_df, on='customer_id', how='left')
submission['prediction'] = submission['article_id'].fillna(fallback)
submission.drop(columns='article_id', inplace=True)
submission.to_csv("submission_ranker.csv", index=False)

print("✅ LGBMRankerベースの submission_ranker.csv を保存しました